In [7]:
pip install numpy soundfile sounddevice matplotlib pandas pydub tqdm librosa seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from IPython.display import Audio 
import pandas as pd
import os
from pathlib import Path
from itertools import product
from pydub import AudioSegment
import numpy as np
from tqdm import tqdm

In [17]:
def load_validated_data(path, max_rows=None):
    """Оптимизированная загрузка данных с фильтрацией колонок"""
    dtype = {
        'client_id': str,
        'path': str,
        'sentence': str,
        'gender': str,
        'up_votes': int,
        'age': str
    }
    return pd.read_csv(
        path, 
        sep='\t', 
        dtype=dtype,
        usecols=list(dtype.keys()),  # Только нужные колонки
        nrows=max_rows
    )

def generate_concatenated_records(
    speakers_df, 
    clips_dir, 
    output_audio_dir, 
    output_meta_dir, 
    target_durations=[5, 15, 30, 60],
    max_source_duration=60,
    max_target_duration=60,
    max_speakers=None,  # Лимит спикеров
    max_files_per_speaker=20  # Лимит файлов на спикера
):
    os.makedirs(output_audio_dir, exist_ok=True)
    os.makedirs(output_meta_dir, exist_ok=True)

    # Фильтрация по качеству записей
    speakers_df = speakers_df[speakers_df['up_votes'] >= 2].copy()
    
    # Ограничение количества спикеров
    unique_speakers = speakers_df['client_id'].unique()
    if max_speakers:
        unique_speakers = unique_speakers[:max_speakers]
    
    for client_id in tqdm(unique_speakers, desc="Обработка спикеров"):
        speaker_files = speakers_df[speakers_df['client_id'] == client_id]
        audio_segments = []
        metadata_segments = []
        current_duration = 0
        
        for _, row in speaker_files.head(max_files_per_speaker).iterrows():
            try:
                audio_path = os.path.join(clips_dir, row['path'])
                audio = AudioSegment.from_file(audio_path)
                duration_sec = len(audio) / 1000
                
                if 0.5 <= duration_sec <= max_source_duration:
                    audio_segments.append(audio)
                    metadata_segments.append({
                        "start": current_duration,
                        "end": current_duration + duration_sec,
                        "text": row['sentence'],
                        "original_file": row['path']
                    })
                    current_duration += duration_sec
                    
                    for target in [t for t in target_durations if t <= max_target_duration]:
                        if current_duration >= target:
                            save_concatenated(
                                audio_segments, metadata_segments,
                                client_id, row['gender'], target,
                                output_audio_dir, output_meta_dir
                            )
                            break
                            
            except Exception as e:
                print(f"\nОшибка в {row['path']}: {str(e)}")
                continue

def save_concatenated(audio_segments, metadata_segments, client_id, gender, target_duration, audio_dir, meta_dir):
    """Сохраняет склеенное аудио и метаданные."""
    concatenated = sum(audio_segments)
    concatenated = concatenated.set_frame_rate(16000).set_channels(1)
    output_name = f"spk_{client_id}_{target_duration}s.wav"
    
    # Сохранение аудио
    concatenated.export(os.path.join(audio_dir, output_name), format="wav")
    
    # Сохранение метаданных
    meta = {
        "speaker_id": client_id,
        "gender": "male" if gender == "male_masculine" else "female",
        "total_duration": round(metadata_segments[-1]['end'], 2),
        "original_files": [seg['original_file'] for seg in metadata_segments],
        "segments": metadata_segments
    }
    
    with open(os.path.join(meta_dir, output_name.replace(".wav", ".json")), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

# Основной блок выполнения
if __name__ == "__main__":
    base_dir = "cv-corpus-21.0-2025-03-14/ru"
    clips_dir = os.path.join(base_dir, "clips")
    validated_path = os.path.join(base_dir, "validated.tsv")

    # Загрузка с ограничением
    df = load_validated_data(validated_path, max_rows=100000)
    print(f"Загружено записей: {len(df)}")

    # Обработка с лимитами
    for gender, speakers_df in [('male', df[df['gender'] == 'male_masculine']),
                               ('female', df[df['gender'] == 'female_feminine'])]:
        print(f"\nОбработка {gender} спикеров...")
        generate_concatenated_records(
            speakers_df,
            clips_dir,
            output_audio_dir="common_voice_processed/audio/concatenated",
            output_meta_dir="common_voice_processed/metadata/concatenated",
            max_speakers=100,  # Обработать только 100 спикеров каждого пола
            max_files_per_speaker=10  # Макс 10 файлов на спикера
        )

Загружено записей: 100000

Обработка male спикеров...


Обработка спикеров: 100%|██████████| 100/100 [00:28<00:00,  3.46it/s]



Обработка female спикеров...


Обработка спикеров: 100%|██████████| 100/100 [00:49<00:00,  2.02it/s]
